# Correlation Heatmap

This example displays the correlation between equity securities in a Plotly heatmap along with the rolling correlation for selected securities in a line chart.

**Additional interactivity:**
- Click on a cell in the heatmap to update the security pair in the rolling correlation line chart.

Run this example via the **Run all** (<i class="fas fa-forward"></i>) button at the top of this window to see the output.

In [3]:
%pip install bql

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement bql (from versions: none)
ERROR: No matching distribution found for bql


In [5]:
# Set up your environment
import bql
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import ipywidgets as ipw

ModuleNotFoundError: No module named 'bql'

In [9]:
# Instantiate connection to BQL
bq = bql.Service()

In [10]:
# Set the universe
universe = bq.univ.members('SENSEX Index')

# Required data items
data_items = {
    'Price': bq.data.px_last(dates=bq.func.range('-3y', '-1d')),
    'Name': bq.data.name()
}

# Build and execute request
request = bql.Request(universe, data_items)
response = bq.execute(request)

# Create DataFrame from response. Price is time series based while
# the Name is a single data point per security. First, extract the time
# series data
df = response.get('Price').df()

# Extract the name from the response and join to the DataFrame
df = df.join(response.get('Name').df())
df = df.sort_values('Name', ascending=True)

In [11]:
# Pivot the DataFrame so each security is a unique column with date as index
df = df.pivot(index='DATE', columns='Name', values='Price')

# Calculate the rolling correlation across all securities
correlation_period = 120
df_correl = df.dropna().pct_change().rolling(correlation_period).corr()
# Select the correlation values for all securities for the final date and
# drop the Date from the multilevel index
heat_map_df = df_correl[df_correl.index.levels[0][-1]:].droplevel(0)

In [12]:
# Mask so that only one half of the heatmap is displayed
mask = np.triu(np.ones_like(heat_map_df, dtype=bool))

# Create the heatmap
fig = go.FigureWidget(
    go.Heatmap(x=heat_map_df.columns, 
               y=heat_map_df.index, 
               z=heat_map_df.mask(mask).fillna(''), 
               colorscale='RdBu',
               zmin=-1,
               zmax=1
              )
)

# Set styling
fig = fig.update_layout(
    title=f'{correlation_period}D Correlation',
    template='plotly_dark', 
    width=800, 
    height=800, 
    xaxis={'categoryarray': sorted(df.columns)},
    yaxis={'categoryarray': sorted(df.columns, reverse=True)}
)

In [13]:
# Select the rolling correlation series for all dates and two securities
line_df = df_correl.loc[(slice(None), df.columns[0]), df.columns[1]].dropna()
# Drop the name part of the multiindex
line_df = line_df.droplevel(1)

# Create the line chart
line_fig = go.FigureWidget()
line_fig.add_trace(
    go.Scatter(
        x=line_df.index, 
        y=line_df.values, 
        name=df.columns[1] + ' vs ' + df.columns[0])
)

# Set some styling
line_fig = line_fig.update_layout(
    title=f'Rolling {correlation_period}D Correlation',
    template='plotly_dark', 
    width=800, 
    height=300, 
    showlegend=True,
    legend=dict(yanchor='bottom', y=-0.4, xanchor='center', x=0.5)
)

def update_line_chart(trace, points, selector):
    """Update the rolling correlation chart on heatmap click"""
    # Which securities were clicked
    security_one = points.xs[0]
    security_two = points.ys[0]
    # Select the rolling correlation for all dates and chosen securities
    line_df = df_correl.loc[(slice(None), security_one), security_two]
    # Drop the name part of the multiindex
    line_df = line_df.droplevel(1)
    # Update the line chart
    with line_fig.batch_update():
        line_fig.data[0].y = line_df.dropna().values
        line_fig.data[0].name = security_two + ' vs ' + security_one

# Bind the heatmap click to the line chart update handler
fig.data[0].on_click(update_line_chart)

# Collect the widgets together
ui_display = ipw.VBox([fig, line_fig])

In [14]:
ui_display

    'data': [{'colorscale': [[0.0, 'rgb(103,0,31)'], [0.1, 'rgb(178,24,43)'],
  …

<span style="color: gray; font-size: 0.8em; font-style: italic; line-height: 2.25">
    <hr>
        While the content, sample projects and code examples available on the BQuant Help Center ("Content") have been reviewed by Bloomberg for reliability and, where relevant, conformance to current market practices, Bloomberg does not guarantee their correctness or completeness and reserves the right to update them from time to time. The Content is made available for illustration purposes only. Customers are solely responsible for the selection of and the use or intended use of the Content, and for verifying their accuracy and adequacy, the resultant output thereof and the assumptions and any other parameters when using such Content.<br>
    <br>
    BQuant services, including BQuant Desktop and BQuant Enterprise, do not express an opinion on the future or projected value of any security and are not research recommendations (i.e., recommendations as to whether or not to “buy”, “sell”, “hold”, or to enter or not to enter into any other transaction involving any specific interest) or a recommendation as to an investment or other strategy.  No aspect of the BQuant services is based on the consideration of a customer’s individual circumstances and information available via the BQuant services should not be considered as information sufficient upon which to base an investment decision.  Customers and their users should determine on their own whether they agree with any code, parameters, inputs, models, formulas and data.  BQuant services should not be construed as tax, accounting, legal or regulatory advice or opinions, or sufficient to satisfy any tax, accounting, legal or regulatory requirements. Customers and their users are solely responsible for the selection of and use of appropriate parameters, inputs, models, formulas and data for meeting their tax, accounting, legal or regulatory requirements.
</span>